# Reversibility: train a 19.4M GPT on 50M tokens three ways

1. **Plain GPT** (a10's model at 10 layers), batch 64 x 256 tokens.
2. **Reversible GPT**, same batch: the layers are wired so backward can rebuild each layer's input from its output, so nothing per layer is stored.
3. **Reversible GPT at the largest batch that fits.**

Before the main runs, the notebook checks every reversible variant's gradients against ordinary autograd, measures bf16 against fp16 against fp32, and screens four ways of making layers reversible (coupling, midpoint, euler, momentum) on 5M tokens each. The screening's winner goes into runs 2 and 3.

It runs on Kaggle (2x T4, the record in the README), Colab (1 GPU) or a Mac. With two GPUs, two runs train side by side. `guide.md` explains every step for a reader new to all of this.

Run it top to bottom. If a session dies, run it again: finished runs are skipped and unfinished ones resume from their last checkpoint.

## 1. Parameters

`SMOKE = True` shrinks every stage to toy size (screening 300K tokens, main runs 2M) to check the whole notebook end to end in about 15 minutes. `GPUS = "auto"` uses every CUDA GPU, or the Apple GPU / CPU when there is none.

In [ ]:
SMOKE = False      # True: every stage at toy size, to check the notebook end to end
GPUS = "auto"      # "auto", a number of GPUs, or a list of CUDA ids such as [0, 1]
OUT_DIR = None     # None: /kaggle/working/a13 on Kaggle, /content/a13 on Colab, runs/ (or runs/smoke) locally
DATA_DIR = None    # None: a Kaggle dataset holding train.bin if attached, else data/fwe50 built by data.py
USE_DRIVE = False  # Colab only: keep OUT_DIR on Google Drive, so a dead session loses nothing
REPO = "https://github.com/nikshrimali/a13-reversibility.git"

## 2. Setup

Finds out where it is running. On Kaggle or Colab the notebook sits alone, so it clones the repo to get `train.py` and the other scripts. Everything heavy runs as a separate process per GPU (see `jobs.py`), so this notebook's own process never holds GPU memory that would skew the memory numbers.

In [ ]:
import glob, json, math, os, subprocess, sys

ON_KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if not os.path.exists("train.py"):
    if not os.path.exists("a13-reversibility/train.py"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("a13-reversibility")
sys.path.insert(0, os.getcwd())
if ON_KAGGLE or ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "pyarrow", "huggingface_hub", "matplotlib"],
                   check=False)

if ON_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if OUT_DIR is None:
    if ON_KAGGLE:
        OUT_DIR = "/kaggle/working/a13"
    elif ON_COLAB:
        OUT_DIR = "/content/drive/MyDrive/a13" if USE_DRIVE else "/content/a13"
    else:
        OUT_DIR = "runs/smoke" if SMOKE else "runs"
os.makedirs(OUT_DIR, exist_ok=True)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local", "| code in", os.getcwd(), "| outputs in", OUT_DIR)

## 3. Hardware

Which GPUs this session has. On Kaggle's 2x T4 the list is `[0, 1]` and two runs train at once, one per GPU. A Mac shows `[None]`: one worker on the Apple GPU.

In [ ]:
import torch
from jobs import Job, bench_job, detect_gpus, probe_job, run_queue, run_script

gpus = detect_gpus(GPUS)
print("torch", torch.__version__)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB, compute capability {p.major}.{p.minor}")
else:
    print("no CUDA GPU; device:", "mps" if torch.backends.mps.is_available() else "cpu")
print("workers:", gpus)
FIRST = gpus[0]

## 4. Data

50M training tokens of FineWeb-Edu and 0.5M held-out tokens, cut with a10's 4,096-token tokenizer. On Kaggle, attach the dataset made from `data/fwe50` and the notebook finds it. Anywhere else, `data.py` downloads and builds it (a few minutes). The documents are read in a fixed order, so every machine gets the same bytes: the held-out file is byte for byte a10's, and a10's whole 25M-token training file is the first half of this one.

In [ ]:
if DATA_DIR is None:
    hits = sorted(glob.glob("/kaggle/input/**/train.bin", recursive=True))
    DATA_DIR = os.path.dirname(hits[0]) if hits else "data/fwe50"
if not os.path.exists(f"{DATA_DIR}/train.bin"):
    subprocess.run([sys.executable, "data.py", "--out", DATA_DIR, "--train-tokens", "50M", "--eval-tokens", "500K",
                    "--tokenizer", "tokenizer.json"], check=True)
meta = json.load(open(f"{DATA_DIR}/meta.json"))
print(DATA_DIR, json.dumps(meta, indent=1))

## 5. Sizes

Every number that changes between the smoke test and the real run, in one place. The real run trains each main model on all 50M tokens once (3,039 steps of 64 x 256 tokens; the last few hundred tokens that do not fill a step are left out).

In [ ]:
if SMOKE:
    S = dict(screen="300K", main="2M", eval_every="250K", screen_eval_every="100K", eval_tokens=65536,
             probe_start=16, probe_cap=128, bench_steps=30)
else:
    S = dict(screen="5M", main="50M", eval_every="2.5M", screen_eval_every="500K", eval_tokens=None,
             probe_start=64, probe_cap=1 << 16, bench_steps=100)
BATCH = 64
COMMON = ["--data", DATA_DIR, "--micro-batch", BATCH]
EVAL = ["--eval-tokens", S["eval_tokens"]] if S["eval_tokens"] else []
R = lambda *p: os.path.join(OUT_DIR, *p)
print(S)

## 6. Gradient check

The reversible backward never sees the layers' stored inputs; it rebuilds them. If the rebuild or the gradient bookkeeping were wrong, the weights would get wrong gradients while the loss looked normal. So each variant runs one batch twice, reversibly and with ordinary autograd, and every parameter's gradient is compared.

Three setups. In **fp64** on the CPU the exact variants must agree to about 1e-14. On the GPU in training precision (fp16 autocast on a T4) both gradients carry rounding error, so each is compared with an fp64 answer from the same weights: a reversible variant costs nothing if its error is no bigger than plain autograd's. The GPU check runs twice, with the streams between layers stored as **fp32** and as **fixed-point integers**.

In [ ]:
if not os.path.exists(R("gradcheck.json")):
    run_script(["gradcheck.py", "--out", R("gradcheck.json")], FIRST)
else:
    print(open(R("gradcheck.json")).read()[:300], "...")
gradcheck = json.load(open(R("gradcheck.json")))

## 7. bf16, fp16 and fp32 speed

A T4 has tensor cores for fp16 (65 TFLOPS on the datasheet) but no hardware path for bf16, which falls back to something near fp32 speed (8.1 TFLOPS). This measures it: a big matmul, then 20 training steps of the plain GPT, in each dtype.

In [7]:
if not os.path.exists(R("precision_speed.json")):
    run_script(["precision.py", "speed", "--out", R("precision_speed.json"), "--batch", 16 if SMOKE else 64], FIRST)
print(json.dumps(json.load(open(R("precision_speed.json"))), indent=1))


dtype   matmul TFLOPS  train tok/s
fp32             2.55       12,370
fp16             2.84       11,606
bf16             2.83        9,852
wrote runs/precision_speed.json
{
 "device": "mps",
 "matmul_tflops": {
  "fp32": 2.548878395104344,
  "fp16": 2.8359386900284145,
  "bf16": 2.8337809269219276
 },
 "step": {
  "fp32": {
   "median_step_s": 1.3244855839984666,
   "tok_per_s": 12370.085562229093
  },
  "fp16": {
   "median_step_s": 1.4117000000005646,
   "tok_per_s": 11605.86526882018
  },
  "bf16": {
   "median_step_s": 1.6629465420010092,
   "tok_per_s": 9852.391274277088
  }
 }
}


## 8. Screening: which way of making layers reversible works?

Seven runs on the same 5M tokens, same batch, same seed, same data order, and the main runs' learning rate schedule squeezed into 5M tokens (50 warmup steps, then cosine down to 1e-4):

| run | wiring | streams stored as |
|---|---|---|
| `gpt` | a10's plain GPT, the reference | n/a |
| `coupling` | y1 = x1 + A(x2), y2 = x2 + M(y1) | fixed-point integers |
| `coupling_fp32` | same | fp32 |
| `midpoint` | x[n+1] = x[n-1] + F(x[n]) | fixed-point integers |
| `midpoint_fp32` | same | fp32 |
| `momentum` | v' = 0.9 v + 0.1 F(x), x' = x + v' | fp32 |
| `euler` | x[n+1] = x[n] + F(x[n]), undone by 3 fixed-point repeats | fp32 |

Every reversible run also logs how far the rebuilt layer inputs drift from the true ones.

In [ ]:
SCREEN = {
    "gpt": ["--arch", "gpt"],
    "coupling": ["--arch", "coupling", "--stream-dtype", "fixed"],
    "coupling_fp32": ["--arch", "coupling", "--stream-dtype", "fp32"],
    "midpoint": ["--arch", "midpoint", "--stream-dtype", "fixed"],
    "midpoint_fp32": ["--arch", "midpoint", "--stream-dtype", "fp32"],
    "momentum": ["--arch", "momentum", "--stream-dtype", "fp32"],
    "euler": ["--arch", "euler", "--stream-dtype", "fp32"],
}
from train import parse_count
screen_steps = parse_count(S["screen"]) // (BATCH * 256)
screen_jobs = [Job(f"screen/{n}", a + COMMON + EVAL + ["--max-tokens", S["screen"], "--eval-every-tokens", S["screen_eval_every"],
                                                       "--recon-every-tokens", S["screen_eval_every"],
                                                       "--warmup", min(50, max(5, screen_steps // 6)), "--sample-tokens", 0],
                   R("screen", n)) for n, a in SCREEN.items()]
run_queue(screen_jobs, gpus)

## 9. Pick the winner

The rule, fixed before the runs:

1. **Gradients**: on the GPU check (cell 6), the variant's error against fp64 is at most 1.5x plain autograd's.
2. **Undo**: the worst rebuilt layer input at the end of the screening is within 1e-3 of the truth.
3. Among the variants that pass both, the **lowest held-out loss** wins, averaged over the last three measurements (4M, 4.5M and 5M tokens) so that one bad step at the very end cannot decide it. If two are within 0.02 of each other, the **faster** one wins.

The smoke test showed why the average matters: every model, the plain GPT included, has the odd single step where the loss jumps by 1 to 4 and recovers on the next step. One of them landed on a variant's final measurement.

In [ ]:
def last(path, kind):
    rows = [json.loads(l) for l in open(path) if f'"type": "{kind}"' in l] if os.path.exists(path) else []
    return rows[-1] if rows else None

def grad_ok(variant, stream):
    for r in gradcheck:
        if r["variant"] == variant and r["stream"] == stream and "rev_vs_fp64" in r:
            return r["rev_vs_fp64"] <= 1.5 * r["plain_vs_fp64"], r["rev_vs_fp64"] / r["plain_vs_fp64"]
    return False, float("nan")

table = []
for n, a in SCREEN.items():
    done = json.load(open(R("screen", n, "done.json")))
    arch = a[1]
    stream = a[3] if len(a) > 3 else None
    rec = last(R("screen", n, "metrics.jsonl"), "recon")
    g_ok, g_ratio = grad_ok(arch, stream) if arch != "gpt" else (True, 1.0)
    r_ok = arch == "gpt" or (rec is not None and rec["max"] <= 1e-3)
    evs = [json.loads(l)["loss"] for l in open(R("screen", n, "metrics.jsonl")) if '"type": "eval"' in l]
    loss = sum(evs[-3:]) / len(evs[-3:])
    table.append(dict(run=n, arch=arch, stream=stream, loss=loss, final=evs[-1], tok_per_s=done["tok_per_s"],
                      grad_ratio=g_ratio, recon=rec["max"] if rec else None,
                      eligible=arch != "gpt" and g_ok and r_ok and loss == loss))
print(f"{'run':<15} {'last 3 avg':>10} {'final':>8} {'tok/s':>9} {'grad err / plain':>17} {'worst undo':>11}  eligible")
for t in table:
    rec = f"{t['recon']:.1e}" if t["recon"] is not None else "-"
    print(f"{t['run']:<15} {t['loss']:>10.4f} {t['final']:>8.4f} {t['tok_per_s']:>9,.0f} {t['grad_ratio']:>17.2f} {rec:>11}  {t['eligible']}")
ok = sorted([t for t in table if t["eligible"]], key=lambda t: t["loss"])
best = ok[0]
close = [t for t in ok if t["loss"] - best["loss"] <= 0.02]
WIN = max(close, key=lambda t: t["tok_per_s"])
WIN_ARGS = SCREEN[WIN["run"]]
json.dump({"table": table, "winner": WIN}, open(R("screening.json"), "w"), indent=2)
print(f"\nwinner: {WIN['run']} (held-out {WIN['loss']:.4f}, {WIN['tok_per_s']:,.0f} tok/s)")

## 10. How each stream format drifts

Takes each reversible variant's screening checkpoint and undoes its layers from the top down, with the streams between layers stored four ways: fixed-point integers, fp32, fp16 and bf16. Fewer bits per number means a bigger rounding step per layer, and the undo has to subtract its way back through all of them. This is the answer to "can we store the streams in bf16?".

In [ ]:
if not os.path.exists(R("precision_drift.json")):
    run_script(["precision.py", "drift", "--runs", R("screen"), "--data", DATA_DIR, "--out", R("precision_drift.json")], FIRST)

## 11. The two main runs: plain GPT and the winner, same batch, 50M tokens each

Same batch (64 x 256), same learning rate schedule (a10's: AdamW, peak 1e-3, 50 warmup steps, cosine to 1e-4), same seed and data order. With two GPUs they train side by side, one per GPU. Held-out loss every 2.5M tokens.

The first measured step of each prints its memory bill: the textbook prediction, what autograd actually kept for backward, and the device's own count.

In [ ]:
main_jobs = [Job("main/gpt", ["--arch", "gpt"] + COMMON + EVAL + ["--max-tokens", S["main"], "--eval-every-tokens", S["eval_every"],
                                                                  "--recon-every-tokens", 0], R("main", "gpt")),
             Job("main/rev", WIN_ARGS + COMMON + EVAL + ["--max-tokens", S["main"], "--eval-every-tokens", S["eval_every"],
                                                         "--recon-every-tokens", S["eval_every"]], R("main", "rev"))]
run_queue(main_jobs, gpus)
for n in ("gpt", "rev"):
    log = open(R("main", n, "log.txt")).read()
    a, b = log.find("What one step costs"), log.find("after backward")
    print(f"===== {n} =====\n" + log[a: log.find("\n", b)] + "\n")

## 12. The largest batch each model can train at

`probe.py` doubles the batch until a step runs out of memory, then bisects to the nearest 8 sequences. Every try is a fresh process running 3 full training steps (forward, backward, AdamW), so the optimizer state is in place and one failure leaves nothing behind. Both models are probed so the gain can be stated: the plain GPT's largest batch against the reversible one's.

In [ ]:
probe_jobs = [probe_job(f"probe/{n}", a + ["--data", DATA_DIR, "--start", S["probe_start"], "--cap", S["probe_cap"]], R("probe", n))
              for n, a in (("gpt", ["--arch", "gpt"]), ("rev", WIN_ARGS))]
run_queue(probe_jobs, gpus)
probes = {n: json.load(open(R("probe", n, "probe.json"))) for n in ("gpt", "rev")}
for n, p in probes.items():
    print(f"{n}: largest batch {p['max_batch']} sequences ({p['max_batch'] * 256:,} tokens per step)"
          + (f", {p['first_oom']} runs out of memory" if p["first_oom"] else ", cap reached"))

## 13. Reversible at its largest batch, 50M tokens

The same 50M tokens in far fewer, far bigger steps. The learning rate grows with the square root of the batch (the usual rule for Adam), capped at 3e-3, and warmup stays at 1.6% of the steps, as in the 64-batch runs. So the loss gap to run 2 mixes two effects: fewer steps, and a different learning rate. If a long run hits out-of-memory that the 3-step probe did not, the batch drops by 8 and the run restarts.

In [ ]:
n_seq = (meta["tokens"]["train"]) // 257
def maxrun_args(b):
    tokens = parse_count(S["main"])
    steps = min(n_seq // b, tokens // (b * 256))
    lr = min(3e-3, 1e-3 * math.sqrt(b / 64))
    return WIN_ARGS + ["--data", DATA_DIR, "--micro-batch", b, "--max-tokens", S["main"], "--lr", f"{lr:.6f}",
                       "--warmup", max(1, round(0.016 * steps)), "--eval-every-tokens", S["eval_every"],
                       "--recon-every-tokens", S["eval_every"]] + EVAL, steps, lr

B_MAX = probes["rev"]["max_batch"]
for attempt in range(3):
    args_, steps, lr = maxrun_args(B_MAX)
    print(f"batch {B_MAX}: {steps} steps, lr {lr:.2e}")
    [j] = run_queue([Job("main/rev_max", args_, R("main", "rev_max"))], gpus[:1])
    if j.returncode != 3:
        break
    subprocess.run(["rm", "-rf", R("main", "rev_max")])
    B_MAX -= 8

## 14. Speed and memory, each run alone on the machine

With two GPUs the main runs shared the machine's 4 CPU cores, which can slow both. These 100-step runs go one at a time on one GPU, with nothing else running, and they are the source of every tok/s and peak-memory number in the README. They also time two configs that were not trained: the plain GPT at its own largest batch, and the winner with fp32 streams, which shows what fixed-point streams cost in speed.

In [ ]:
bench = {
    "gpt_b64": ["--arch", "gpt", "--micro-batch", BATCH],
    "rev_b64": WIN_ARGS + ["--micro-batch", BATCH],
    "rev_bmax": WIN_ARGS + ["--micro-batch", B_MAX],
    "gpt_bmax": ["--arch", "gpt", "--micro-batch", probes["gpt"]["max_batch"]],
}
if "--stream-dtype" in WIN_ARGS and WIN_ARGS[WIN_ARGS.index("--stream-dtype") + 1] == "fixed":
    bench["rev_fp32_b64"] = WIN_ARGS[:WIN_ARGS.index("--stream-dtype")] + ["--stream-dtype", "fp32", "--micro-batch", BATCH]
run_queue([bench_job(f"bench/{n}", a + ["--data", DATA_DIR, "--bench-steps", S["bench_steps"]], R("bench", n))
           for n, a in bench.items()], gpus[:1])

## 15. Results

The table the README quotes. Final loss is the held-out loss on all 0.5M eval tokens after the last step. Speed and memory come from the solo runs of cell 14. MFU counts 6N FLOPs per token (useful work only); HFU adds the FLOPs the reversible backward spends recomputing blocks.

In [ ]:
rows = []
for name, run, b in (("plain GPT, batch 64", "gpt", "gpt_b64"), ("reversible, batch 64", "rev", "rev_b64"),
                     ("reversible, max batch", "rev_max", "rev_bmax")):
    d = json.load(open(R("main", run, "done.json")))
    s = json.load(open(R("bench", b, "bench.json")))
    rows.append(dict(run=name, arch=d["arch"], batch=d["micro_batch"], steps=d["steps"], tokens=d["tokens"],
                     final_loss=d["final_eval_loss"], tok_per_s=s["tok_per_s"], peak_alloc_gb=s["peak_alloc"] / 1e9,
                     peak_reserved_gb=s["peak_reserved"] / 1e9, mfu=s["mfu"], hfu=s["hfu"],
                     train_minutes=d["wall_seconds"] / 60, device=s["device"]))
extra = {n: json.load(open(R("bench", n, "bench.json"))) for n in bench if n not in ("gpt_b64", "rev_b64", "rev_bmax")}
summary = {"rows": rows, "extra_bench": extra, "winner": WIN, "probes": {n: {k: p[k] for k in ("max_batch", "first_oom")}
                                                                         for n, p in probes.items()}}
json.dump(summary, open(R("summary.json"), "w"), indent=2)
pct = lambda v: f"{100 * v:.1f}%" if v is not None else "n/a"
print("| run | batch | steps | final held-out loss | tok/s | peak allocated | peak reserved | MFU | HFU |")
print("|---|---|---|---|---|---|---|---|---|")
for r in rows:
    print(f"| {r['run']} | {r['batch']} | {r['steps']:,} | {r['final_loss']:.4f} | {r['tok_per_s']:,.0f} | "
          f"{r['peak_alloc_gb']:.2f} GB | {r['peak_reserved_gb']:.2f} GB | {pct(r['mfu'])} | {pct(r['hfu'])} |")
for n, s in extra.items():
    print(f"| ({n}, timing only) | {s['micro_batch']} | | | {s['tok_per_s']:,.0f} | {s['peak_alloc'] / 1e9:.2f} GB | "
          f"{s['peak_reserved'] / 1e9:.2f} GB | {pct(s['mfu'])} | {pct(s['hfu'])} |")

## 16. Plots

Written to `plots/` inside the output folder: screening curves, the main loss curves, memory against batch size, peak memory, speed, and the drift of each stream format.

In [ ]:
PLOTS = os.path.join(OUT_DIR, "plots")
subprocess.run([sys.executable, "plot.py", "--runs", OUT_DIR, "--out", PLOTS], check=True)
from IPython.display import Image, display
for f in ("main_loss", "screening", "probe", "memory", "speed", "recon"):
    if os.path.exists(f"{PLOTS}/{f}.png"):
        display(Image(f"{PLOTS}/{f}.png"))

## 17. What the models write

Each main run ends by continuing the prompt "The most important thing about learning is" for 200 tokens. At 19.4M parameters and 50M tokens, expect grammatical-looking English that does not stay on topic.

In [ ]:
for n in ("gpt", "rev", "rev_max"):
    p = R("main", n, "sample.txt")
    if os.path.exists(p):
        print(f"===== {n} =====\n{open(p).read()}")